In [1]:
from datasets import load_dataset
import pandas as pd
import re


In [2]:
ds_raw  = load_dataset("MahsaVafaie/BZKopen", "raw")
ds_norm = load_dataset("MahsaVafaie/BZKopen", "normalized")

DATE_COLS = ["ApplicantBirthDate", "VictimBirthDate", "VictimDeathDate"]

def hf_to_df(ds):
    dfs = []
    for split in ["train", "validation", "test"]:
        df = ds[split].to_pandas().drop(columns=["image"])
        df["split"] = split
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

raw_df  = hf_to_df(ds_raw)
norm_df = hf_to_df(ds_norm)

print(f"Total rows: {len(raw_df)}")
print(f"Splits:     {raw_df['split'].value_counts().to_dict()}")
raw_df[["split"] + DATE_COLS].head()


Total rows: 516
Splits:     {'train': 361, 'test': 78, 'validation': 77}


,split,ApplicantBirthDate,VictimBirthDate,VictimDeathDate
0,train,17.4.1819,,
1,train,6.6.1819,,
2,train,,25.8.1829,
3,train,,13.3.50,
4,train,,8.7.50,


In [3]:
pairs = pd.concat([
    raw_df[DATE_COLS].rename(columns=lambda c: c + "_raw"),
    norm_df[DATE_COLS].rename(columns=lambda c: c + "_norm"),
], axis=1)
mask = raw_df[DATE_COLS].apply(lambda col: col.str.len() > 0).any(axis=1)

In [ ]:
GERMAN_MONTHS = {
    'januar': 1,  'jan': 1,   'january': 1,  'j': 1,
    'jänner': 1,  'janner': 1,
    'februar': 2, 'feb': 2,   'febr': 2,     'february': 2,
    'maerz': 3,   'märz': 3,  'mar': 3,      'mär': 3,    'march': 3,
    'april': 4,   'apr': 4,
    'mai': 5,     'may': 5,
    'juni': 6,    'jun': 6,   'june': 6,     'jani': 6,
    'juli': 7,    'jul': 7,   'july': 7,
    'august': 8,  'aug': 8,   'ug': 8,       'augl': 8,
    'september': 9, 'sep': 9, 'sept': 9,
    'oktober': 10, 'okt': 10, 'oct': 10,     'october': 10, 'k': 10, 'ct':10,
    'november': 11, 'nov': 11, 'novl': 11,
    'dezember': 12, 'dez': 12, 'dec': 12,    'december': 12, 'dezemb': 12,
    'elul': 6,   
}

ROMAN = {
    'i': 1, 'ii': 2, 'iii': 3, 'iv': 4, 'v': 5, 'vi': 6,
    'vii': 7, 'viii': 8, 'ix': 9, 'x': 10, 'xi': 11, 'xii': 12,
    'xiii': 13,  
}

NON_DATE_TOKENS = {
    'unbekannt', 'deportiert', 'verstorben', 'verst', 'deportation',
    'umgekommen', 'gestorben', 'beide', 'unk',
}

SEASON_TO_MONTH = {'sommer': 8, 'frühling': 4, 'herbst': 10, 'winter': 1}

_EMPTY_RAW = {'nan', 'none', 'null', '', '-', '?'}

_DATE_TOKEN = re.compile(r'\b(\d{1,2})[.\-/]+\s*(\d{1,2})[.\-/]+\s*(\d{2,4})\b')

# Two 4-digit years joined by connector — take the latest
_ODER_PAT = re.compile(
    r'\b(\d{4})\s*(?:oder|or|bis|und|,)\s*(\d{4})\b', re.IGNORECASE
)

# OCR digit-for-letter fixes applied before all pattern matching
_OCR_FIXES = [
    (re.compile(r'\b0([Kk][Tt])\b'), r'O\1'),   # 0kt → Okt
    (re.compile(r'\b1([Vv])\b'),     r'I\1'),   # 1V  → IV
    (re.compile(r'\b8([Uu][Ll])\b'), r'J\1'),   # 8ul → Jul
]

_UML = r'A-Za-z\u00c4\u00e4\u00d6\u00f6\u00dc\u00fc\u00df'



class _UncertainYear(int):
    """Int subclass marking a guessed century for an ambiguous 2-digit year.

    level: 1 = slightly uncertain (~), 2 = unlikely (~~), 3 = very unlikely (~~~)
    """
    def __new__(cls, value, level=1):
        obj = int.__new__(cls, value)
        obj.level = level
        return obj
    def __add__(self, other):
        return _UncertainYear(int.__add__(self, other), self.level)
    def __radd__(self, other):
        return _UncertainYear(int.__radd__(self, other), self.level)




def _ocr_fix(s):
    for pat, repl in _OCR_FIXES:
        s = pat.sub(repl, s)
    return s


def _parse_roman(s):
    """Parse any Roman numeral string (case-insensitive). Returns 0 on failure."""
    vals = {'i': 1, 'v': 5, 'x': 10, 'l': 50, 'c': 100}
    s = s.lower()
    if not s or not all(c in vals for c in s):
        return 0
    result, prev = 0, 0
    for ch in reversed(s):
        v = vals[ch]
        result += v if v >= prev else -v
        prev = v
    return result


# Year each compensation office layout class started processing claims.
# These are placeholders (all set to 1946) to be confirmed with archivists.
# The value is used to compute the age a person born in the 1800s would have
# reached by that year, which determines the plausibility of the 1800s century guess.
_OFFICE_START_YEAR = {
    # Frühe Phase 
    "BY-HB-Frühe Phase":                          1946,
    "HE-Frühe-Phase":                             1946,
    "NI-Frühe-Phase":                             1946,
    "NRW-Frühe-Phase":                            1946,
    "RLP-Frühe-Phase":                            1946,
    # Hauptphase 
    "BW-Hauptphase":                              1946,
    "BY-BE-Hauptphase":                           1946,
    "HE-Hauptphase":                              1946,
    "HH-NI-NRW-SH-Hauptphase":                   1946,
    "HH-NI-NRW-SH-Hauptphase (abweichend 1)":    1946,
    "HH-NI-NRW-SH-Hauptphase (abweichend 2)":    1946,
    "HH-NI-NRW-SH-Hauptphase (abweichend 3)":    1946,
    "HH-NI-NRW-SH-Hauptphase (abweichend 4)":    1946,
    "RLP-Hauptphase":                             1946,
    "RLP-Hauptphase (abweichend 1)":              1946,
    "RLP-Hauptphase (abweichend 2)/Saarland":     1946,
    "NRW-Köln-Art-V":                             1946,
    "NRW-Köln-Härtefonds":                        1946,
    "NRW-LRB":                                    1946,
    "NRW-Innenministerium":                       1946,
    # Spätphase 
    "BY-Spätphase":                               1946,
    # Special/index-card layouts
    "ABC-Karten":                                 1946,
    "Hinweiskarte (Mainz/Neustadt)":              1946,
    "Hinweiskarte_schwarzes_Datum":               1946,
    "Tabellen-Typ":                               1946,
    "Tabellen-Typ (abweichend)":                  1946,
}
_DEFAULT_OFFICE_START = 1946   # fallback for unknown layout classes



# Fixed reference year for VictimBirthDate: earliest year of Nazi persecution.
# A victim must have been alive in 1933; the persecution era ran 1933-1945.
_PERSECUTION_REF_YEAR = 1933


def _extract_year(raw):
    """Extract a plausible year from a raw date string, or None.

    Returns a 4-digit year when one is found.  When the string contains only a
    2-digit year at the end of a date-like pattern.
    """
    if not raw:
        return None
    hit = re.search(r'\b(1[89]\d{2}|20\d{2})\b', raw)
    if hit:
        return int(hit.group(1))
    # 2-digit year at the end of a date pattern — treat as 1900s (death dates are always 1900s)
    hit = re.search(r'(?:^|[.\-/\s])(\d{2})$', raw.strip())
    if hit:
        return 1900 + int(hit.group(1))
    return None


def _century(yy, col, ctx=None):
    """Resolve a 2-digit year to a century using column name and optional context dict.

    The ambiguous ranges and unambiguous boundaries are defined by archival rules:
        ApplicantBirthDate: yy >= 55 → 1800s | yy <= 33 → 1900s | 34–54 ambiguous
        VictimBirthDate:    yy >= 45 → 1800s | yy <= 33 → 1900s | 34–44 ambiguous

    Within the ambiguous ranges, uncertainty levels are derived empirically from
    filename ground-truth birth-year distributions 

        ApplicantBirthDate 34–54:
            34–47  ~~~  0–2%  of records are 1800s — very unlikely
            48–49  ~~   9–13% of records are 1800s —  unlikely
            50–54  ~    20–71% of records are 1800s — genuinely possible

        VictimBirthDate 34–44:
            34–42  ~~~  0–1%  of records are 1800s — very unlikely
            43–44  ~~   2–3%  of records are 1800s — unlikely
          

    VictimBirthDate also uses death_year cross-reference to resolve unambiguously
    when one century implies an implausible age at death (outside 0–105 years).
    """
    if 'DeathDate' in (col or ''):
        return 1900

    death_year = (ctx or {}).get('death_year')

    # VictimBirthDate: cross-reference with death year when available.
    if death_year is not None and col == 'VictimBirthDate':
        age_if_1900s = death_year - (1900 + yy)
        age_if_1800s = death_year - (1800 + yy)
        plausible_1900s = 0 <= age_if_1900s <= 105
        plausible_1800s = 0 <= age_if_1800s <= 105
        if plausible_1900s and not plausible_1800s:
            return 1900
        if plausible_1800s and not plausible_1900s:
            return 1800
        # Both plausible — fall through to field-specific heuristic

    if col == 'ApplicantBirthDate':
        if yy >= 55: return 1800                       # unambiguous 1800s
        if yy <= 33: return 1900                       # unambiguous 1900s
        # Ambiguous range 34–54 — empirical sub-divisions:
        if yy >= 50: return _UncertainYear(1900, 1)    # 36% of records are 1800s
        if yy >= 48: return _UncertainYear(1900, 2)    # 11% of records are 1800s
        return          _UncertainYear(1900, 3)        #  0% of records are 1800s

    if col == 'VictimBirthDate':
        if yy >= 45: return 1800                       # unambiguous 1800s
        if yy <= 33: return 1900                       # unambiguous 1900s
        # Ambiguous range 34–44 — empirical sub-divisions:
        if yy >= 43: return _UncertainYear(1900, 2)    #  2% of records are 1800s
        return          _UncertainYear(1900, 3)        #  0% of records are 1800s

    # Fallback for other columns (no specific rules)
    return 1800 if yy >= 50 else 1900


def _fix_component(v):
    if v > 31 and v >= 10:
        rev = int(str(v)[::-1])
        if 1 <= rev <= 31:
            return rev
    return v


def _to_iso(day, month, year, partial=False):
    try:
        d, m, y = _fix_component(int(day)), int(month), int(year)
        m = _fix_component(m)
        if m > 12 and d <= 12:
            d, m = m, d
        if 1 <= m <= 12 and 1 <= d <= 31:
            return f'{y:04d}-{m:02d}-{d:02d}'
        if partial:
            return f'{y:04d}-{m:02d}-01' if 1 <= m <= 12 else f'{y:04d}-01-01'
        return '', 0
    except (ValueError, TypeError):
        if partial:
            try: return f'{int(year):04d}-01-01'
            except: pass
        return '', 0


def _find_all_date_tokens(s):
    return [(h.group(1), h.group(2), h.group(3)) for h in _DATE_TOKEN.finditer(s)]


def _resolve_year(yr_str, col, ctx=None):
    yr_int = int(yr_str)
    if yr_int >= 100:
        return yr_int
    century = _century(yr_int, col, ctx)
    return century + yr_int


def _month_from_str(s):
    key = s.lower().rstrip('.')
    return GERMAN_MONTHS.get(key) or SEASON_TO_MONTH.get(key)


def _normalize_single(s, col=None, ctx=None):
    s = s.strip()

    # Unwrap if entire string is parenthesized: "(1885-06-07)", "(1892, 8, 7, )"
    if s.startswith('(') and s.endswith(')') and '(' not in s[1:-1]:
        return _normalize_single(s[1:-1].strip(), col, ctx)

    # Preserve year in inline parens before stripping them: "25. Aug. (1920)"
    paren_yr = re.search(r'\((\d{4})\)', s)
    s = re.sub(r'\s*\([^)]*\)', '', s).strip()
    if paren_yr and not re.search(r'\b\d{4}\b', s):
        s = s.rstrip('.').strip() + ' ' + paren_yr.group(1)

    s = re.sub(r'\.$', '', s).strip()
    if not s or s.lower() in _EMPTY_RAW:
        return '', 0

    s = _ocr_fix(s)

    # Already ISO
    hit = re.match(r'^(\d{4})-(\d{2})-(\d{2})', s)
    if hit:
        return f'{hit.group(1)}-{hit.group(2)}-{hit.group(3)}'

    lower = s.lower()

    # Non-date status tokens
    if any(tok in lower for tok in NON_DATE_TOKENS):
        tokens = _find_all_date_tokens(s)
        if tokens:
            d, mo, yr = tokens[0]
            resolved = _resolve_year(yr, col, ctx)
            return _to_iso(d, mo, (resolved.level if isinstance(resolved, _UncertainYear) else 0), resolved)
        month_map = {**GERMAN_MONTHS, **SEASON_TO_MONTH}
        for mon_str, mo in month_map.items():
            if mon_str in lower:
                hit2 = re.search(r'\b(\d{4})\b', s)
                if hit2: return f'{hit2.group(1)}-{mo:02d}-01'
        return '', 0

    if s == '?':
        return '', 0
    hit = re.match(r'^\?\.\s*(\d{4})$', s)
    if hit:
        return f'{hit.group(1)}-01-01'

    # Two 4-digit years joined by connector — take the latest
    hit = _ODER_PAT.search(s)
    if hit:
        return f'{max(int(hit.group(1)), int(hit.group(2))):04d}-01-01'

    # "YYYY u. <rest>" — strip leading year+connector, normalize remainder
    hit = re.match(r'^\d{4}\s+u\.\s+(.+)$', s)
    if hit:
        return _normalize_single(hit.group(1), col)

    # "YYYY, <rest>" — leading year with comma; normalize rest with year injected
    hit = re.match(r'^(\d{4}),\s*(.+)$', s)
    if hit:
        yr, rest = hit.group(1), hit.group(2).strip().rstrip('.')
        hit2 = re.match(r'^(\d{1,2})[.\-/,]+(\d{1,2})$', rest)
        if hit2:
            resolved = _resolve_year(yr, col, ctx)
            return _to_iso(hit2.group(1), hit2.group(2), resolved, partial=True), (resolved.level if isinstance(resolved, _UncertainYear) else 0)
        candidate = _normalize_single(f'{rest} {yr}', col)
        if candidate:
            return candidate

    # "Jahr" keyword — date context marker; extract year
    if re.search(r'\bjahr\b', lower):
        hit = re.search(r'\b(\d{4})\b', s)
        if hit: return f'{hit.group(1)}-01-01'

    # D.M.YYYY or D.M.YY — accepts . - / , as separators
    hit = re.match(r'^(\d{1,2})[.\-/,]+\s*(\d{1,2})[.\-/,]+\s*(\d{2,4})$', s)
    if hit:
        resolved = _resolve_year(hit.group(3), col, ctx)
        return _to_iso(hit.group(1), hit.group(2), resolved, partial=True), (resolved.level if isinstance(resolved, _UncertainYear) else 0)

    # D M. YYYY — space before numeric month, separator after (also handles oversized day)
    hit = re.match(r'^(\d+)\s+(\d{1,2})[.\-]+\s*(\d{2,4})$', s)
    if hit:
        d_raw = int(hit.group(1))
        if d_raw > 31:
            last2 = int(str(d_raw)[-2:])
            day = last2 if 1 <= last2 <= 31 else 1
        else:
            day = d_raw
        resolved = _resolve_year(hit.group(3), col, ctx)
        return _to_iso(day, hit.group(2), resolved, partial=True), (resolved.level if isinstance(resolved, _UncertainYear) else 0)

    # Oversized day (3-4 digits).M.YYYY — take last 2 digits of first component
    hit = re.match(r'^(\d{3,4})[.\-/]+(\d{1,2})[.\-/]+(\d{4})$', s)
    if hit:
        last2 = int(hit.group(1)[-2:])
        return _to_iso(last2 if 1 <= last2 <= 31 else 1, hit.group(2), hit.group(3), partial=True)

    # YYYY.M.D
    hit = re.match(r'^(\d{4})[.\-/]+(\d{1,2})[.\-/]+(\d{1,2})$', s)
    if hit:
        return _to_iso(hit.group(3), hit.group(2), hit.group(1), partial=True)

    # MM/DD/YYYY US
    hit = re.match(r'^(\d{1,2})/(\d{1,2})/(\d{4})$', s)
    if hit:
        return _to_iso(hit.group(2), hit.group(1), hit.group(3))

    # D.MYY — merged: "3.275" = day=3, month=2, year=75
    hit = re.match(r'^(\d{1,2})\.(\d)(\d{2})$', s)
    if hit:
        resolved = _resolve_year(hit.group(3), col, ctx)
        return _to_iso(hit.group(1), hit.group(2), resolved, partial=True), (resolved.level if isinstance(resolved, _UncertainYear) else 0)

    # D.Roman.YYYY — e.g. "16.XIII.00" → 1900-12-16, "15.IV.33" → 1933-04-15,
    #                     "0.XI.1899" → 1899-11-01, "11.XX.1915" → 1915-12-11
    hit = re.match(r'^(\d{1,2})[\s.\-/,]+([IVXivxLlCc]+)[\s.\-/,]+(\d{2,4})$', s)
    if hit:
        mo_r = _parse_roman(hit.group(2))
        if mo_r:
            resolved = _resolve_year(hit.group(3), col, ctx)
            return _to_iso(hit.group(1), min(mo_r, 12), resolved, partial=True), (resolved.level if isinstance(resolved, _UncertainYear) else 0)

    # Roman.Roman.YYYY — e.g. "II.II.1940" → 1940-02-02
    hit = re.match(r'^([IVXivx]+)[.\-/]+([IVXivx]+)[.\-/]+(\d{2,4})$', s)
    if hit:
        day_r = _parse_roman(hit.group(1))
        mo_r  = _parse_roman(hit.group(2))
        if day_r and mo_r:
            resolved = _resolve_year(hit.group(3), col, ctx)
            return _to_iso(day_r, mo_r, (resolved.level if isinstance(resolved, _UncertainYear) else 0), resolved)

    # Roman. D. YYYY — Roman=month, numeric=day: "II. 6. 1936" → 1936-02-06
    hit = re.match(r'^([IVXivx]+)[.\-\s]+(\d{1,2})[.\-\s]+(\d{2,4})$', s)
    if hit:
        mo_r = _parse_roman(hit.group(1))
        if mo_r:
            resolved = _resolve_year(hit.group(3), col, ctx)
            return _to_iso(hit.group(2), mo_r, (resolved.level if isinstance(resolved, _UncertainYear) else 0), resolved)

    # Roman. Month YYYY — Roman=day, text=month: "II. Sept. 1937" → 1937-09-02
    hit = re.match(rf'^([IVXivx]+)[.\-\s]+([{_UML}]+)\.?[.\-\s,]*(\d{{2,4}})$', s)
    if hit:
        day_r = _parse_roman(hit.group(1))
        mo = _month_from_str(hit.group(2))
        if day_r and mo:
            resolved = _resolve_year(hit.group(3), col, ctx)
            return _to_iso(day_r, mo, (resolved.level if isinstance(resolved, _UncertainYear) else 0), resolved)

    # D1/D2 M. YYYY — two alternative days, numeric month: "2/8 3. 1930" → 1930-03-02
    hit = re.match(r'^(\d{1,2})/\d+\s+(\d{1,2})[.\-]+\s*(\d{2,4})$', s)
    if hit:
        mo = int(hit.group(2))
        if 1 <= mo <= 12:
            resolved = _resolve_year(hit.group(3), col, ctx)
            return _to_iso(hit.group(1), mo, (resolved.level if isinstance(resolved, _UncertainYear) else 0), resolved)

    # D1/D2. Month YYYY — two alternative days, text month: "14/26. März 1880" → 1880-03-14
    hit = re.match(rf'^(\d{{1,2}})/\d{{1,2}}[.\s]+([{_UML}]+)\.?[.\-\s,]*(\d{{2,4}})$', s)
    if hit:
        mo = _month_from_str(hit.group(2))
        if mo:
            resolved = _resolve_year(hit.group(3), col, ctx)
            return _to_iso(hit.group(1), mo, (resolved.level if isinstance(resolved, _UncertainYear) else 0), resolved)

    # D/D YYYY — ambiguous fraction; extract year only: "5/17 1944" → 1944-01-01
    hit = re.match(r'^(\d{1,2})/(\d{1,2})\s+(\d{4})$', s)
    if hit:
        return f'{hit.group(3)}-01-01'

    # Ordinal day + Month + YYYY: "19th March 1913" → 1913-03-19
    hit = re.match(rf'^(\d{{1,2}})(?:st|nd|rd|th)[.,\s]+([{_UML}]+)\.?\s*(\d{{4}})$', s)
    if hit:
        mo = _month_from_str(hit.group(2))
        if mo:
            resolved = _resolve_year(hit.group(3), col, ctx)
            return _to_iso(hit.group(1), mo, (resolved.level if isinstance(resolved, _UncertainYear) else 0), resolved)

    # Month oder Month YYYY — take the second (later) month: "Juli oder Aug. 1937"
    hit = re.match(rf'^([{_UML}]+)\.?\s+(?:oder|or)\s+([{_UML}]+)\.?\s*(\d{{4}})$', s)
    if hit:
        mo = _month_from_str(hit.group(2))
        if mo: return f'{hit.group(3)}-{mo:02d}-01'

    # D Month YYYY — includes / in after-month separator for cases like "1.Jul/1883"
    hit = re.match(
        rf'^(\d{{1,2}})[.\-\s,]+([{_UML}]+)\.?[.\-\s,/]*(\d{{2,4}})$', s
    )
    if hit:
        mo = _month_from_str(hit.group(2))
        if mo:
            resolved = _resolve_year(hit.group(3), col, ctx)
            return _to_iso(hit.group(1), mo, resolved, partial=True), (resolved.level if isinstance(resolved, _UncertainYear) else 0)

    # Im/In Month YYYY
    hit = re.match(rf'^[Ii][mn]\s+([{_UML}]+)\.?\s*(\d{{4}})$', s)
    if hit:
        mo = _month_from_str(hit.group(1))
        if mo: return f'{hit.group(2)}-{mo:02d}-01'

    # Month D, YYYY — "March 28, 1942", "Sept. 14, 1898", "April 23, 1892"
    hit = re.match(rf'^([{_UML}]+)\.?\s+(\d{{1,2}}),?\s*(\d{{4}})$', s)
    if hit:
        mo = _month_from_str(hit.group(1))
        if mo: return _to_iso(hit.group(2), mo, hit.group(3))

    # Month YYYY or YY — "Feb. 42", "August 1942", "Augl 1901"
    hit = re.match(rf'^([{_UML}]+)\.?\s*(\d{{2,4}})$', s)
    if hit:
        mo = _month_from_str(hit.group(1))
        if mo:
            resolved = _resolve_year(hit.group(2), col, ctx)
            return f'{resolved:04d}-{mo:02d}-01', (resolved.level if isinstance(resolved, _UncertainYear) else 0)

    # YYYY D. Month — day before text month: "1889 21. Elul"
    hit = re.match(rf'^(\d{{4}})\s+(\d{{1,2}})[.\s]+([{_UML}]+)\.?$', s)
    if hit:
        mo = _month_from_str(hit.group(3))
        if mo: return _to_iso(hit.group(2), mo, hit.group(1))

    # YYYY Month YYYY — spurious leading year: "1940 März 1939" → 1939-03-01
    hit = re.match(rf'^(\d{{4}})\s+([{_UML}]+)\.?\s*(\d{{4}})$', s)
    if hit:
        mo = _month_from_str(hit.group(2))
        if mo: return f'{hit.group(3)}-{mo:02d}-01'

    # YYYY Month D — "1894 Aug. 18"
    hit = re.match(rf'^(\d{{4}})\s+([{_UML}]+)\.?\s*(\d{{1,2}})$', s)
    if hit:
        mo = _month_from_str(hit.group(2))
        if mo: return _to_iso(hit.group(3), mo, hit.group(1))

    # YYYY. Roman. D — "1897. XII. 23"
    hit = re.match(r'^(\d{4})[.\-\s]+([IVXivx]+)[.\-\s]+(\d{1,2})$', s)
    if hit:
        mo_r = _parse_roman(hit.group(2))
        if mo_r: return _to_iso(hit.group(3), min(mo_r, 12), hit.group(1))

    # YYYY/Month or YYYY Month — "1882/November", "1875 März"
    hit = re.match(rf'^(\d{{4}})[/\s]+([{_UML}]+)\.?$', s)
    if hit:
        mo = _month_from_str(hit.group(2))
        if mo: return f'{hit.group(1)}-{mo:02d}-01'

    # YYYY-Month text — "1903-Sept."
    hit = re.match(rf'^(\d{{4}})[.\-]([{_UML}]+)\.?$', s)
    if hit:
        mo = _month_from_str(hit.group(2))
        if mo: return f'{hit.group(1)}-{mo:02d}-01'

    # YYYY im Month — "1923 im April"
    hit = re.match(rf'^(\d{{4}})\s+[Ii]m\s+([{_UML}]+)\.?$', s)
    if hit:
        mo = _month_from_str(hit.group(2))
        if mo: return f'{hit.group(1)}-{mo:02d}-01'

    # Ende YYYY
    hit = re.match(r'^[Ee]nde\s+(\d{4})$', s)
    if hit: return f'{hit.group(1)}-12-01'

    # M/YYYY or M.YYYY — "11/1906" → 1906-11-01
    hit = re.match(r'^(\d{1,2})[./](\d{4})$', s)
    if hit:
        m = int(hit.group(1))
        if 1 <= m <= 12: return f'{hit.group(2)}-{m:02d}-01'

    # YYYY-MM partial
    hit = re.match(r'^(\d{4})[.\-/](\d{1,2})$', s)
    if hit:
        m = int(hit.group(2))
        if 1 <= m <= 12: return f'{hit.group(1)}-{m:02d}-01'

    # YYYY only
    hit = re.match(r'^(\d{4})$', s)
    if hit: return f'{hit.group(1)}-01-01'

    # M. YY or M. YYYY: "11. 1859", "12.75", "2.1906"
    hit = re.match(r'^(\d{1,2})\.\s*(\d{2}|\d{4})$', s)
    if hit:
        resolved = _resolve_year(hit.group(2), col, ctx)
        return f'{resolved:04d}-{int(hit.group(1)):02d}-01', (resolved.level if isinstance(resolved, _UncertainYear) else 0)

    # YYYY-YYYY year range: take first
    hit = re.match(r'^(\d{4})[.\-/]+(\d{4})$', s)
    if hit: return f'{hit.group(1)}-01-01'

    # D.D. Month YYYY — strip leading double-numeric OCR noise: "19.36. Nov. 1936"
    hit = re.match(rf'^\d+[./]+\d+[./\s]+([{_UML}]+)\.?\s*(\d{{2,4}})$', s)
    if hit:
        mo = _month_from_str(hit.group(1))
        if mo:
            resolved = _resolve_year(hit.group(2), col, ctx)
            return f'{resolved:04d}-{mo:02d}-01', (resolved.level if isinstance(resolved, _UncertainYear) else 0)

    # Last resort: first D.M.Y token
    tokens = _find_all_date_tokens(s)
    if tokens:
        d, mo, yr = tokens[0]
        resolved = _resolve_year(yr, col, ctx)
        return _to_iso(d, mo, resolved, partial=True), (resolved.level if isinstance(resolved, _UncertainYear) else 0)

    # Year-only fallback: extract any plausible 4-digit year from noisy strings
    # e.g. "2. M. 1882", "ca. 1900", "187.1901", "22.1-4/2 1911"
    hit = re.search(r'\b(\d{4})\b', s)
    if hit:
        yr = int(hit.group(1))
        if 1800 <= yr <= 2024:
            return f'{yr:04d}-01-01'

    return ''


_MULTI_LABELED = re.compile(r'[ab1-9]\)\s*')
_TWO_DATES = re.compile(
    r'^(\d{1,2}[.\-/]+\d{1,2}[.\-/]+\d{2,4})\s+(\d{1,2}[.\-/]+\d{1,2}[.\-/]+\d{2,4})$'
)


def _as_pair(x):
    """Ensure _normalize_single output is always a (iso_str, level) tuple."""
    if isinstance(x, tuple):
        return x[0] or '', x[1] if len(x) > 1 else 0
    return x or '', 0


def normalize_date(raw, col=None, ctx=None):
    """Normalise raw date string to ISO 8601. Multiple dates -> 'D1 ; D2'.
    """
    if not raw:
        return '', 0
    s = raw.strip()
    if not s or s.lower() in _EMPTY_RAW:
        return '', 0
    s_clean = re.sub(r'\s*\([^)]*\)', '', s).strip()
    if not s_clean:
        s_clean = s.strip('()').strip()

    parts = [p.strip() for p in _MULTI_LABELED.split(s_clean) if p.strip()]
    if len(parts) > 1:
        pairs = [_as_pair(_normalize_single(p, col, ctx)) for p in parts]
        results = [(iso, lvl) for iso, lvl in pairs if iso]
        if not results: return '', 0
        return ' ; '.join(iso for iso, _ in results), max(lvl for _, lvl in results)

    if ';' in s_clean:
        parts = [p.strip() for p in s_clean.split(';') if p.strip()]
        pairs = [_as_pair(_normalize_single(p, col, ctx)) for p in parts]
        results = [(iso, lvl) for iso, lvl in pairs if iso]
        if not results: return '', 0
        return ' ; '.join(iso for iso, _ in results), max(lvl for _, lvl in results)

    hit = _TWO_DATES.match(s_clean)
    if hit:
        pairs = [_as_pair(_normalize_single(hit.group(1), col, ctx)),
                 _as_pair(_normalize_single(hit.group(2), col, ctx))]
        results = [(iso, lvl) for iso, lvl in pairs if iso]
        if not results: return '', 0
        return ' ; '.join(iso for iso, _ in results), max(lvl for _, lvl in results)

    return _as_pair(_normalize_single(s, col, ctx))



In [5]:
def eval_normalization(split='validation'):
    mask = raw_df['split'] == split
    raw_split  = raw_df[mask].reset_index(drop=True)
    norm_split = norm_df[mask].reset_index(drop=True)

    rows = []
    for i in range(len(raw_split)):
        # Use VictimDeathDate to cross-reference birth date uncertainty within the same record
        death_raw = raw_split.at[i, 'VictimDeathDate'] if 'VictimDeathDate' in raw_split.columns else None
        death_year = _extract_year(death_raw) if death_raw else None
        for col in DATE_COLS:
            raw_val = raw_split.at[i, col]
            gt_val  = norm_split.at[i, col]
            if not raw_val:
                continue
            pred_val, _uncertainty = normalize_date(raw_val, col=col, ctx={'death_year': death_year})
            rows.append({
                'split': split,
                'record_idx': i,
                'col': col, 'raw': raw_val,
                'pred': pred_val, 'gt': gt_val,
                'uncertainty': _uncertainty,
                'correct': pred_val == gt_val,
            })
    df = pd.DataFrame(rows)
    n            = len(df)
    n_correct    = df['correct'].sum()
    n_u1 = df['uncertainty'].eq(1).sum()
    n_u2 = df['uncertainty'].eq(2).sum()
    n_u3 = df['uncertainty'].eq(3).sum()
    n_uncertain  = n_u1 + n_u2 + n_u3
    has_gt       = df['gt'].notna() & (df['gt'] != '')
    n_gt         = has_gt.sum()
    n_correct_gt = df.loc[has_gt, 'correct'].sum()
    print(
        f"[{split}]  all: {n_correct}/{n} ({100*n_correct/n:.1f}%)"
        f"  |  with GT: {n_correct_gt}/{n_gt} ({100*n_correct_gt/n_gt:.1f}%)"
        f"  |  uncertain(~={n_u1},~~={n_u2},~~~={n_u3})"
    )
    return df

val_results   = eval_normalization('validation')
train_results = eval_normalization('train')
test_results  = eval_normalization('test')


[validation]  all: 79/91 (86.8%)  |  with GT: 78/84 (92.9%)  |  uncertain(~=0,~~=0,~~~=0)
[train]  all: 436/469 (93.0%)  |  with GT: 423/434 (97.5%)  |  uncertain(~=0,~~=0,~~~=0)
[test]  all: 97/106 (91.5%)  |  with GT: 91/94 (96.8%)  |  uncertain(~=0,~~=0,~~~=0)


In [6]:
errors = test_results[~test_results["correct"]].copy()
print(f"Errors on test: {len(errors)}")
errors[["record_idx", "raw", "pred", "gt", "col"]]


Errors on test: 9


,record_idx,raw,pred,gt,col
8,4,1944,1944-01-01,,VictimDeathDate
24,13,24.6.1941,1941-06-24,,VictimDeathDate
26,15,1.1.76,1876-01-01,1876-01-04,VictimBirthDate
27,15,gestorben 19.2.47,0000-02-19,1947-02-19,VictimDeathDate
34,21,1880 gen. Dat. fehlt - 15.7.1966 <UNK>,0000-07-15,1966-07-15,ApplicantBirthDate
36,22,3.8.1944,1944-08-03,,VictimDeathDate
38,23,25.8.1940,1940-08-25,,VictimDeathDate
70,51,26. März 1903,1903-03-26,,ApplicantBirthDate
105,77,6.3.1941,1941-03-06,,VictimDeathDate


In [ ]:
# Layout-class based evaluation using matched_offices_by_fullname_2.csv

import json, re
import pandas as pd
from collections import defaultdict
from pathlib import Path

BZK_DATA_DIR = Path("/home/bzk-data")

# Layout classes where the date of interest is VictimBirthDate
VICTIM_BIRTHDATE_CLASSES = {
    "BY-HB-Frühe Phase",
    "HE-Frühe-Phase",
    "NRW-Frühe-Phase",
}

# Layout classes to skip entirely
IGNORE_CLASSES = {
    "Gerichtsurteile",
    "Gelbe-Hinweiskarte",
    "Rückseite_Weitere Namen",
    "Siehe-auch-Hinweiskarte",
}

# raw values that carry no date info and should not count as 'unparsed'
_SKIP_RAW = {'nan', 'none', 'null', '', '-', '?'}

# Filenames must follow YYYY_MM_DD_seq_seq -- others are non-person records
_FNAME_PAT = re.compile(r'^(\d{4})_(\d{2})_(\d{2})_\d+_\d+$')


def filename_to_gt(stem):
    m = _FNAME_PAT.match(stem)
    if not m:
        return None
    yyyy, mm, dd = int(m.group(1)), int(m.group(2)), int(m.group(3))
    if mm == 0:
        return f"{yyyy:04d}-01-01"
    if dd == 0:
        return f"{yyyy:04d}-{mm:02d}-01"
    return f"{yyyy:04d}-{mm:02d}-{dd:02d}"


# Load CSV mapping and build year : {filename: (gt, layout_class, date_col)}
mapping_df = pd.read_csv(BZK_DATA_DIR / "matched_offices_by_fullname_2.csv")
mapping_df = mapping_df[~mapping_df["Layout class"].isin(IGNORE_CLASSES)].copy()

by_year = defaultdict(dict)

for _, row in mapping_df.iterrows():
    fname = row["Filename"]
    layout_class = row["Layout class"]
    stem = fname.rsplit(".", 1)[0]
    gt = filename_to_gt(stem)
    if gt is None:
        continue
    year = stem.split("_")[0]
    date_col = "VictimBirthDate" if layout_class in VICTIM_BIRTHDATE_CLASSES else "ApplicantBirthDate"
    by_year[year][fname] = (gt, layout_class, date_col)

print(f"Valid files in mapping: {sum(len(v) for v in by_year.values()):,}")
print(f"Years spanned         : {min(by_year)} – {max(by_year)}")

# Load matching records from JSONL files
rows = []
for year, fname_map in sorted(by_year.items()):
    jsonl = BZK_DATA_DIR / f"{year}.jsonl"
    if not jsonl.exists():
        continue
    with open(jsonl, encoding="utf-8") as fh:
        for line in fh:
            rec   = json.loads(line)
            fname = rec.get("filename", "")
            if fname not in fname_map:
                continue
            gt, layout_class, date_col = fname_map[fname]
            raw  = (rec.get(date_col) or '').strip()
            # Build context: cross-reference death year + office era for uncertainty reduction
            death_raw = (rec.get('VictimDeathDate') or '').strip()
            death_year = _extract_year(death_raw)
            pred, uncertainty_level = normalize_date(raw, col=date_col, ctx={'death_year': death_year}) if raw and raw.lower() not in _SKIP_RAW else ('', 0)
            rows.append({
                "region":            layout_class,
                "date_col":          date_col,
                "filename":          fname,
                "gt":                gt,
                "raw":               raw,
                "pred":              pred,
                "correct":           pred == gt,
                "uncertain":         pred.startswith('~'),
                "uncertainty_level": uncertainty_level,
                "empty":             pred == "" and bool(raw) and raw.lower() not in _SKIP_RAW,
            })

regional_df = pd.DataFrame(rows)
print(f"Records matched       : {len(regional_df):,}")
print(f"  with raw date value : {(regional_df['raw'] != '').sum():,}")
print(f"  no raw value        : {(regional_df['raw'] == '').sum():,}")
regional_df.head(5)


Valid files in mapping: 1,163,263
Years spanned         : 1819 – 1998
Records matched       : 1,163,347
  with raw date value : 1,150,595
  no raw value        : 12,752


,region,date_col,filename,gt,raw,pred,correct,uncertain,uncertainty_level,empty
0,BY-BE-Hauptphase,ApplicantBirthDate,1819_06_06_1_0.jpg,1819-06-06,6.6.1819,1819-06-06,True,False,0,False
1,BY-BE-Hauptphase,ApplicantBirthDate,1842_09_27_1_0.jpg,1842-09-27,,,False,False,0,False
2,NRW-LRB,ApplicantBirthDate,1846_02_26_1_0.jpg,1846-02-26,26.7.1886,1886-07-26,False,False,0,False
3,BY-HB-Frühe Phase,VictimBirthDate,1850_03_13_1_0.jpg,1850-03-13,13.3.50,1850-03-13,True,False,0,False
4,BY-BE-Hauptphase,ApplicantBirthDate,1853_04_20_2_0.jpg,1853-04-20,20.4.1853,1853-04-20,True,False,0,False


In [8]:
# Evaluate normalization on regional data

def eval_regional(df, label="ALL"):
    sub = df[df["raw"] != ""].copy()
    n           = len(sub)
    if n == 0:
        print(f"[{label}]  no records with raw date value")
        return
    n_correct   = sub["correct"].sum()
    n_uncertain = sub["uncertain"].sum()
    # Per-level uncertainty breakdown
    if "uncertainty_level" in sub.columns:
        n_u1 = (sub["uncertainty_level"] == 1).sum()
        n_u2 = (sub["uncertainty_level"] == 2).sum()
        n_u3 = (sub["uncertainty_level"] == 3).sum()
        unc_detail = f"uncertain(~={n_u1},~~={n_u2},~~~={n_u3})"
    else:
        unc_detail = f"uncertain(~)={n_uncertain:>5,}"
    n_empty     = sub["empty"].sum()
    print(
        f"[{label:<45}]  n={n:>7,}  "
        f"correct={n_correct:>7,} ({100*n_correct/n:5.1f}%)  "
        f"{unc_detail}  unparsed={n_empty:>5,}"
    )

eval_regional(regional_df, "ALL regions")
print()
for layout_class in sorted(regional_df["region"].unique()):
    sub = regional_df[regional_df["region"] == layout_class]
    eval_regional(sub, layout_class)

[ALL regions                                  ]  n=1,150,595  correct=1,088,322 ( 94.6%)  uncertain(~=236,~~=145,~~~=12169)  unparsed=  130

[ABC-Karten                                   ]  n=    794  correct=    747 ( 94.1%)  uncertain(~=0,~~=0,~~~=14)  unparsed=    0
[BW-Hauptphase                                ]  n= 47,041  correct= 46,329 ( 98.5%)  uncertain(~=1,~~=2,~~~=87)  unparsed=    4
[BY-BE-Hauptphase                             ]  n=221,558  correct=215,391 ( 97.2%)  uncertain(~=45,~~=35,~~~=389)  unparsed=    5
[BY-HB-Frühe Phase                            ]  n=    614  correct=    358 ( 58.3%)  uncertain(~=0,~~=0,~~~=5)  unparsed=    0
[BY-Spätphase                                 ]  n=     76  correct=     72 ( 94.7%)  uncertain(~=0,~~=0,~~~=1)  unparsed=    0
[HE-Frühe-Phase                               ]  n=     40  correct=     33 ( 82.5%)  uncertain(~=0,~~=0,~~~=0)  unparsed=    0
[HE-Hauptphase                                ]  n= 79,943  correct= 76,775 ( 96.0%) 

In [9]:
# Browse errors and edge-cases on regional data 
reg_with_raw = regional_df[regional_df["raw"] != ""].copy()

errors_reg    = reg_with_raw[~reg_with_raw["correct"]]
uncertain_reg = reg_with_raw[reg_with_raw["uncertain"]]
empty_reg     = reg_with_raw[reg_with_raw["empty"]]

print(f"Wrong predictions : {len(errors_reg):,}")
print(f"Uncertain (~)     : {len(uncertain_reg):,}")
print(f"Could not parse   : {len(empty_reg):,}")
print()
display(errors_reg[["region","filename","raw","pred","gt"]].query('region == "NI-Frühe-Phase"'))

Wrong predictions : 62,273
Uncertain (~)     : 0
Could not parse   : 130



,region,filename,raw,pred,gt
239099,NI-Frühe-Phase,1897_08_29_94_0.jpg,29.8.1899,1899-08-29,1897-08-29
258641,NI-Frühe-Phase,1898_11_16_4_0.jpg,16.11.1998,1998-11-16,1898-11-16


In [10]:
empty_reg[["region","filename","raw","pred","gt"]]

,region,filename,raw,pred,gt
11272,HE-Hauptphase,1876_08_05_5_0.jpg,5.0.0,,1876-08-05
17302,HH-NI-NRW-SH-Hauptphase,1877_06_04_18_0.jpg,4.6.19XX,,1877-06-04
18552,HH-NI-NRW-SH-Hauptphase,1878_11_08_2_0.jpg,8. M. 18,,1878-11-08
31239,Hinweiskarte_schwarzes_Datum,1880_11_04_10_0.jpg,11.8.0,,1880-11-04
34738,HH-NI-NRW-SH-Hauptphase,1881_08_14_3_0.jpg,14 de agosto 1.881,,1881-08-14
...,...,...,...,...,...
1099457,HH-NI-NRW-SH-Hauptphase,1935_07_11_18_0.jpg,117.35,,1935-07-11
1110272,NRW-LRB,1937_04_19_19_0.jpg,1910111937,,1937-04-19
1116301,HH-NI-NRW-SH-Hauptphase (abweichend 4),1937_09_02_13_0.jpg,02.09.12937,,1937-09-02
1122062,NRW-LRB,1938_04_19_5_0.jpg,190411978,,1938-04-19


In [21]:
# ── Sanity-check
cases = [
    ("11. 1859",              "ApplicantBirthDate", "1859-11-01"),
    ("5.0.66",                "ApplicantBirthDate", "1866-01-01"),
    ("75.70.1953",            "ApplicantBirthDate", "1953-07-01"),
    ("1886-1865",             "ApplicantBirthDate", "1886-01-01"),
    ("nan",                   "ApplicantBirthDate", ""),
    ("2-II-1870",             "ApplicantBirthDate", "1870-02-02"),
    ("1875 März",             "ApplicantBirthDate", "1875-03-01"),
    ("1974.3.1",              "ApplicantBirthDate", "1974-03-01"),
    ("1874.8.74",             "ApplicantBirthDate", "1874-08-01"),
    ("Im Juli 1942",          "VictimDeathDate",    "1942-07-01"),
    ("1944-11",               "VictimDeathDate",    "1944-11-01"),
    ("22-Oktober-1946",       "VictimBirthDate",    "1946-10-22"),
    ("1943 oder 1944",        "VictimDeathDate",    "1944-01-01"),
    ("1.0kt.1871",            "ApplicantBirthDate", "1871-10-01"),
    ("1894 Aug. 18",          "ApplicantBirthDate", "1894-08-18"),
    ("12.75",                 "ApplicantBirthDate", "1875-12-01"),
    ("3.275",                 "ApplicantBirthDate", "1875-02-03"),
    ("1. J. 1877",            "ApplicantBirthDate", "1877-01-01"),
    ("11. ug. 1942",          "VictimDeathDate",    "1942-08-11"),
    ("18 IX, 1942",           "VictimDeathDate",    "1942-09-18"),
    ("Feb. 42",               "VictimDeathDate",    "1942-02-01"),
    ("March 28, 1942",        "VictimDeathDate",    "1942-03-28"),
    ("Jänner 1877",           "ApplicantBirthDate", "1877-01-01"),
    ("1914.10.1877",          "ApplicantBirthDate", "1877-10-14"),
    ("1940 März 1939",        "VictimDeathDate",    "1939-03-01"),
    ("1, III. 1940",          "VictimDeathDate",    "1940-03-01"),
    ("II.II.1940",            "VictimDeathDate",    "1940-02-02"),
    ("116.7.1936",            "ApplicantBirthDate", "1936-07-16"),
    ("1913.7.1936",           "ApplicantBirthDate", "1936-07-13"),
    ("1934 u. 1. Aug. 1938",  "VictimDeathDate",    "1938-08-01"),
    ("II. Sept. 1937",        "VictimDeathDate",    "1937-09-02"),
    ("14/26. März 1880",      "ApplicantBirthDate", "1880-03-14"),
    ("1882/November",         "ApplicantBirthDate", "1882-11-01"),
    ("II. 6. 1936",           "VictimDeathDate",    "1936-02-06"),
    ("19.36. Nov. 1936",      "VictimDeathDate",    "1936-11-01"),
    ("2. Novl 1883",          "ApplicantBirthDate", "1883-11-02"),
    ("7,6,1885",              "ApplicantBirthDate", "1885-06-07"),
    ("(1885-06-07)",          "ApplicantBirthDate", "1885-06-07"),
    ("1888, 15. März",        "ApplicantBirthDate", "1888-03-15"),
    ("1889 21. Elul",         "ApplicantBirthDate", "1889-06-21"),
    ("1897. XII. 23",         "ApplicantBirthDate", "1897-12-23"),
    ("1890, 31.1.",            "ApplicantBirthDate", "1890-01-31"),
    ("1893, Sept. 14",        "ApplicantBirthDate", "1893-09-14"),
    ("8, 7, 1898",            "ApplicantBirthDate", "1898-07-08"),
    ("16.XIII.00",            "ApplicantBirthDate", "1900-12-16"),
    ("3,10. 1898",            "ApplicantBirthDate", "1898-10-03"),
    ("24. K. 1900",           "ApplicantBirthDate", "1900-10-24"),
    ("1903, 7. Oktober",      "ApplicantBirthDate", "1903-10-07"),
    ("8 10. 1905",            "ApplicantBirthDate", "1905-10-08"),
    ("22,2,1906",             "ApplicantBirthDate", "1906-02-22"),
    ("12 Dezemb 1915",        "ApplicantBirthDate", "1915-12-12"),
    ("25. Jani 1916",         "ApplicantBirthDate", "1916-06-25"),
    ("81 4. 29",              "VictimDeathDate",    "1929-04-01"),
    ("24 3-1935",             "ApplicantBirthDate", "1935-03-24"),
    ("1. Jahr 1891",          "ApplicantBirthDate", "1891-01-01"),
    ("5/17 1944",             "VictimDeathDate",    "1944-01-01"),
    ("Augl 1901",             "ApplicantBirthDate", "1901-08-01"),
    ("1903-Sept.",            "ApplicantBirthDate", "1903-09-01"),
    ("1920, April",           "ApplicantBirthDate", "1920-04-01"),
    ("25. Aug. (1920)",        "ApplicantBirthDate", "1920-08-25"),
    ("1923 im April",         "ApplicantBirthDate", "1923-04-01"),
    ("2/8 3. 1930",           "ApplicantBirthDate", "1930-03-02"),
    ("15.1V.33",              "VictimDeathDate",    "1933-04-15"),
    ("74. März 1897",         "ApplicantBirthDate", "1897-03-01"),
    ("0. Januar 1914",        "ApplicantBirthDate", "1914-01-01"),
    ("April 23, 1892",        "ApplicantBirthDate", "1892-04-23"),
    ("Sept. 14, 1898",        "ApplicantBirthDate", "1898-09-14"),
    ("1.8ul/1883",            "ApplicantBirthDate", "1883-07-01"),
    ("2. M. 1882",            "ApplicantBirthDate", "1882-01-01"),
    ("(1892, 8, 7, )",        "ApplicantBirthDate", "1892-07-08"),
    ("ca. 1900",              "ApplicantBirthDate", "1900-01-01"),
    ("187.1901",              "ApplicantBirthDate", "1901-01-01"),
    ("22.1-4/2 1911",         "ApplicantBirthDate", "1911-01-01"),
    ("11.XX.1915",            "ApplicantBirthDate", "1915-12-11"),
    ("25.Jan. oder Feb.1919", "ApplicantBirthDate", "1919-01-01"),
    ("Juli oder Aug. 1937",   "VictimDeathDate",    "1937-08-01"),
    ("0.XI.1899",             "ApplicantBirthDate", "1899-11-01"),
    ("11/1906",               "ApplicantBirthDate", "1906-11-01"),
    ("19th March 1913",       "ApplicantBirthDate", "1913-03-19"),
]
print(f"{'raw':<30} {'col':<22} {'pred':<15} {'expected':<15} ok")
print("-" * 90)
all_ok = True
for raw, col, expected in cases:
    pred = normalize_date(raw, col=col)
    ok   = pred[0] == expected
    all_ok = all_ok and ok
    print(f"{raw:<30} {col:<22} {pred[0]:<15} {expected:<15} {'✓' if ok else '✗  ← FAIL'}")

raw                            col                    pred            expected        ok
------------------------------------------------------------------------------------------
11. 1859                       ApplicantBirthDate     1859-11-01      1859-11-01      ✓
5.0.66                         ApplicantBirthDate     1866-01-01      1866-01-01      ✓
75.70.1953                     ApplicantBirthDate     1953-07-01      1953-07-01      ✓
1886-1865                      ApplicantBirthDate     1886-01-01      1886-01-01      ✓
nan                            ApplicantBirthDate                                     ✓
2-II-1870                      ApplicantBirthDate     1870-02-02      1870-02-02      ✓
1875 März                      ApplicantBirthDate     1875-03-01      1875-03-01      ✓
1974.3.1                       ApplicantBirthDate     1974-03-01      1974-03-01      ✓
1874.8.74                      ApplicantBirthDate     1874-08-01      1874-08-01      ✓
Im Juli 1942                

---
## Multi-source date resolution

Priority chain per date field for each record:
1. **Metadata** — archivist-curated Excel files (`metadata/`) linked via name + birth-date merge
2. **Filename** — ground-truth date encoded in the image filename (`YYYY_MM_DD_seq_n`)
3. **Normalization pipeline** — `normalize_date()` on the raw OCR value

Each resolved date carries a `source` tag: `metadata | filename | normalized | unparsed | missing`.

In [ ]:
# Build merge index: filename -> metadata dates


import re, json
from pathlib import Path
from typing import Optional
import pandas as pd

_META_FILES = {
    "480":    Path("metadata/findbuch-480-bearbeitet.xlsx"),
    "EL-350": Path("metadata/findbuch-EL-350-I-bearbeitet.xlsx"),
    "F-196":  Path("metadata/findbuch-F-196-gesamt-bearbeitet.xlsx"),
    "Wue-33": Path("metadata/findbuch-Wue-33-T-1-bearbeitet.xlsx"),
}

def _norm_date_iso(s) -> Optional[str]:
    if pd.isna(s) or str(s).strip() in ('', 'NaN', 'nan'): return None
    s = str(s).strip()
    m = re.match(r'^(\d{4})-(\d{2})-(\d{2})', s)
    if m: return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"
    m = re.match(r'^(\d{1,2})\.(\d{1,2})\.(\d{4})', s)
    if m: return f"{m.group(3)}-{int(m.group(2)):02d}-{int(m.group(1)):02d}"
    m = re.match(r'^(\d{4})$', s)
    if m: return f"{m.group(1)}-01-01"
    return None

def _norm_name(s) -> str:
    return str(s).strip().lower() if not pd.isna(s) else ''

def _load_metadata() -> pd.DataFrame:
    dfs = []
    for src, path in _META_FILES.items():
        if not path.exists():
            print(f'  [skip] {path} not found')
            continue
        raw = pd.read_excel(path, header=None)
        df  = raw.iloc[2:].reset_index(drop=True)
        df.columns = raw.iloc[0].tolist()
        bd_col = (
            next((c for c in df.columns if 'geburt' in str(c).lower()
                  and 'datum' in str(c).lower() and 'format' in str(c).lower()), None)
            or next((c for c in df.columns if 'geburt' in str(c).lower()
                     and 'datum' in str(c).lower()), None)
        )
        sd_col = (
            next((c for c in df.columns if 'sterbe' in str(c).lower()
                  and 'datum' in str(c).lower() and 'format' in str(c).lower()), None)
            or next((c for c in df.columns if 'sterbe' in str(c).lower()
                     and 'datum' in str(c).lower()), None)
        )
        dfs.append(pd.DataFrame({
            'source':          src,
            'Bestellsignatur': df['Bestellsignatur'],
            'meta_last':       df['Nachname(n)'].apply(_norm_name),
            'meta_first':      df.get('Vorname(n)', pd.Series(dtype=str)).apply(_norm_name),
            'meta_bd':         (df[bd_col] if bd_col else pd.Series(dtype=str)).apply(_norm_date_iso),
            'meta_sd':         (df[sd_col] if sd_col else pd.Series(dtype=str)).apply(_norm_date_iso),
        }))
    meta = pd.concat(dfs, ignore_index=True)
    return meta[meta['meta_bd'].notna() & meta['meta_last'].ne('')]


def _build_merge_index(bzk_dir: Path, meta_df: pd.DataFrame) -> dict:
    """
    filename -> {source, Bestellsignatur, meta_sd_iso}
    Joins on victim last name + birth date + first name (when both sides non-empty).
    Used only to recover VictimDeathDate from metadata; birth dates are excluded
    because the join key already includes birth date (circular recovery).
    """
    rows = []
    for jsonl in sorted(bzk_dir.glob('*.jsonl')):
        with open(jsonl, encoding='utf-8') as fh:
            for line in fh:
                rec = json.loads(line)
                rows.append({
                    'filename':  rec.get('filename', ''),
                    'vic_last':  _norm_name(rec.get('VictimLastName')),
                    'vic_first': _norm_name(rec.get('VictimFirstName')),
                    'vic_bd':    _norm_date_iso(rec.get('VictimBirthDate')),
                })
    bzk = pd.DataFrame(rows)

    # Allow first name to be missing on either side (OCR may have dropped it),
    # but when both sides have a value they must agree.
    def _first_ok(a, b): return a == b or a == '' or b == ''

    sub = bzk[bzk['vic_bd'].notna() & bzk['vic_last'].ne('')]
    m   = pd.merge(sub, meta_df, left_on=['vic_last', 'vic_bd'],
                   right_on=['meta_last', 'meta_bd'], how='inner')
    m   = m[m.apply(lambda r: _first_ok(r['vic_first'], r['meta_first']), axis=1)]

    index = {}
    for _, row in m.iterrows():
        fn = row['filename']
        if fn not in index:
            index[fn] = {'source': row['source'],
                         'Bestellsignatur': row['Bestellsignatur'],
                         'meta_sd_iso': row['meta_sd']}
    print(f"Merge matches: {len(m):,}  →  {len(index):,} unique filenames")
    return index



In [ ]:
# Build filename index: filename: (gt_date_iso, layout_class, date_field)
#
# _LAYOUT_FILENAME_FIELD: maps each layout class to the field the filename date encodes.
#   None  = role order undefined: filename unreliable,
#           skip it 
#
# _LAYOUT_ABSENT_FIELDS: fields structurally absent from a layout.
#
# Source: BZK archival documentation on card layout order.

_LAYOUT_FILENAME_FIELD = {
    # filename = VictimBirthDate
    "BY-HB-Frühe Phase":               "VictimBirthDate",
    "BY-eigener-Typ (abweichend 1)":   "VictimBirthDate",
    "HH-Frühe-Phase":                  "VictimBirthDate",
    "HE-Frühe-Phase":                  "VictimBirthDate",   # explicitly stated in docs
    "NRW-Frühe-Phase":                 "VictimBirthDate",

    # filename = ApplicantBirthDate
    "BY-BE-Hauptphase":                            "ApplicantBirthDate",
    "BY-eigener-Typ":                              "ApplicantBirthDate",
    "BY-eigener-Typ (abweichend 2)":               "ApplicantBirthDate",
    "BY-Spätphase":                                "ApplicantBirthDate",
    "BY-Spätphase (abweichend 1)":                 "ApplicantBirthDate",
    "HH-NI-NRW-SH-Hauptphase":                    "ApplicantBirthDate",
    "HH-NI-NRW-SH-Hauptphase (abweichend 1)":     "ApplicantBirthDate",
    "HH-NI-NRW-SH-Hauptphase (abweichend 2)":     "ApplicantBirthDate",
    "HH-NI-NRW-SH-Hauptphase (abweichend 3)":     "ApplicantBirthDate",
    "HH-NI-NRW-SH-Hauptphase (abweichend 4)":     "ApplicantBirthDate",
    "NI-Frühe-Phase":                              "ApplicantBirthDate",   # explicitly stated in docs
    "Tabellen-Typ":                                "ApplicantBirthDate",
    "Tabellen-Typ (abweichend)":                   "ApplicantBirthDate",
    "HB-Spätphase":                                "ApplicantBirthDate",
    "HE-Hauptphase":                               "ApplicantBirthDate",
    "NRW-LRB":                                     "ApplicantBirthDate",
    "NRW-Köln-Art-V":                              "ApplicantBirthDate",
    "NRW-Köln-Härtefonds":                         "ApplicantBirthDate",
    "RLP-Hauptphase":                              "ApplicantBirthDate",
    "RLP-Hauptphase (abweichend 1)":               "ApplicantBirthDate",
    "RLP-Hauptphase (abweichend 2) / Saarland":    "ApplicantBirthDate",
    "RLP-Hauptphase (abweichend 3)":               "ApplicantBirthDate",
    "RLP-Hauptphase (abweichend 4)":               "ApplicantBirthDate",
    "Hinweiskarte_schwarzes_Datum":                "ApplicantBirthDate",
    "BW-Hauptphase":                               "ApplicantBirthDate",

    # No predefined role order = skip
    "NRW-Innenministerium":                           None,
    "Auskünfte_Statistisches_Landesamt_NRW":          None,
    "Auskünfte_Statistisches_Landesamt_NRW (abweichend)": None,
    "RLP-Frühe-Phase":                                None,
    "ABC-Karten":                                     None,
    "Suchkarte-Hinweiskarte":                         None,
}

# Fields structurally absent from specific layout classes (never on the card).
# Source: archival documentation.
_LAYOUT_ABSENT_FIELDS = {
    # HE-Frühe-Phase: "kein Feld für das Geburtsdatum vorgesehen" for Antragstellende
    "HE-Frühe-Phase":      {"ApplicantBirthDate"},
    # NRW-LRB and NRW-Köln-Härtefonds: only Anspruchsberechtigter, no victim role
    "NRW-LRB":             {"VictimBirthDate"},
    "NRW-Köln-Härtefonds": {"VictimBirthDate"},
}

_IGNORE_CLASSES = {
    "Gerichtsurteile", "Gelbe-Hinweiskarte",
    "Rückseite_Weitere Namen", "Siehe-auch-Hinweiskarte",
}

_FNAME_PAT2 = re.compile(r'^(\d{4})_(\d{2})_(\d{2})_\d+_\d+$')

def _stem_to_gt(stem: str) -> Optional[str]:
    m = _FNAME_PAT2.match(stem)
    if not m: return None
    yyyy, mm, dd = int(m.group(1)), int(m.group(2)), int(m.group(3))
    if mm == 0: return f'{yyyy:04d}-01-01'
    if dd == 0: return f'{yyyy:04d}-{mm:02d}-01'
    return f'{yyyy:04d}-{mm:02d}-{dd:02d}'

_map_df = pd.read_csv(BZK_DATA_DIR / 'matched_offices_by_fullname_2.csv')
_map_df = _map_df[~_map_df['Layout class'].isin(_IGNORE_CLASSES)]

filename_index = {}
skipped_ambiguous = 0
for _, row in _map_df.iterrows():
    fname = row['Filename']
    lc    = row['Layout class']
    stem  = fname.rsplit('.', 1)[0]
    gt    = _stem_to_gt(stem)
    if gt is None: continue
    field = _LAYOUT_FILENAME_FIELD.get(lc)
    if field is None:          # ambiguous layout — filename unreliable
        skipped_ambiguous += 1
        continue
    filename_index[fname] = (gt, lc, field)

print(f'Filename index: {len(filename_index):,} entries  (skipped {skipped_ambiguous:,} ambiguous)')
print(f'  -> ApplicantBirthDate: {sum(1 for _,_,f in filename_index.values() if f=="ApplicantBirthDate"):,}')
print(f'  -> VictimBirthDate:    {sum(1 for _,_,f in filename_index.values() if f=="VictimBirthDate"):,}')


Filename index: 1,110,916 entries  (skipped 52,347 ambiguous)
  -> ApplicantBirthDate: 1,096,677
  -> VictimBirthDate:    14,239


In [ ]:
# Empirical birth-year analysis: determine uncertainty sub-divisions within the ambiguous ranges.
#
# Archival rules define the ambiguous zones:
#   ApplicantBirthDate: yy 34-54  (yy>=55 → unambiguous 1800s, yy<=33 → unambiguous 1900s)
#   VictimBirthDate:    yy 34-44  (yy>=45 → unambiguous 1800s, yy<=33 → unambiguous 1900s)
#
# compute the 1800s probability within those zones from filename GT.


# For example, yy = 52:
#
# Count records with birth year 1852: 5
# Count records with birth year 1952: 19
# P(1800s | yy=52) = 5 / (5+19) = 21%

import re, pandas as pd, warnings
warnings.filterwarnings('ignore')
from pathlib import Path

def _extract_year_int(s):
    m = re.search(r'(\d{4})', str(s))
    return int(m.group(1)) if m else None

# Filename GT birth years
_fname_years = [_extract_year_int(gt) for gt, _, _ in filename_index.values()]
_fname_years = pd.Series([y for y in _fname_years if y and 1800 <= y < 1960])


for col, lo, hi in [("ApplicantBirthDate", 34, 54), ("VictimBirthDate", 34, 44)]:
    print(f"=== {col} — ambiguous range yy {lo}–{hi} ===")
    print(f"  {'yy':>4}  {'1800s':>7}  {'1900s':>7}  {'total':>7}  {'% 1800s':>8}  level")
    t1800, t1900 = 0, 0
    for yy in range(lo, hi + 1):
        n1800 = int((_fname_years == 1800 + yy).sum())
        n1900 = int((_fname_years == 1900 + yy).sum())
        total = n1800 + n1900
        pct   = 100 * n1800 / total if total else 0.0
        t1800 += n1800
        t1900 += n1900
        if col == 'ApplicantBirthDate':
            level = '~' if yy >= 50 else '~~' if yy >= 48 else '~~~'
        else:  # VictimBirthDate
            level = '~~' if yy >= 43 else '~~~'
        print(f"  {yy:>4}  {n1800:>7}  {n1900:>7}  {total:>7}  {pct:>7.1f}%  {level}")
    ttotal = t1800 + t1900
    tpct   = 100 * t1800 / ttotal if ttotal else 0.0
    print(f"  {'tot':>4}  {t1800:>7}  {t1900:>7}  {ttotal:>7}  {tpct:>7.1f}%")
    print()


# How many records actually require a century guess?
def _raw_yy(s):
    """Return the 2-digit year from a raw date string, or None if year is 4-digit or absent."""
    if not s or str(s).strip() in ('', 'nan', 'NaN', '-', '?'):
        return None
    s = str(s).strip()
    # 4-digit year 
    if re.search(r'\b\d{4}\b', s):
        return None
    # D.M.YY patterns
    m = re.search(r'[./\-,\s](\d{2})$', s)
    if m:
        return int(m.group(1))
    # Bare YY
    m = re.match(r'^(\d{2})$', s)
    if m:
        return int(m.group(1))
    return None

_AMBIG = {
    'ApplicantBirthDate': (34, 54),
    'VictimBirthDate':    (34, 44),
}


BZK_DATA_DIR = Path("/home/bzk-data")

counts = {field: {'total_parseable': 0, 'ambiguous': 0} for field in _AMBIG}

for jsonl in sorted(BZK_DATA_DIR.glob('*.jsonl')):
    with open(jsonl, encoding='utf-8') as fh:
        for line in fh:
            rec = json.loads(line)
            for field, (lo, hi) in _AMBIG.items():
                raw = rec.get(field) or ''
                if not raw or str(raw).strip().lower() in ('', 'nan', '-', '?'):
                    continue
                counts[field]['total_parseable'] += 1
                yy = _raw_yy(raw)
                if yy is not None and lo <= yy <= hi:
                    counts[field]['ambiguous'] += 1

print("=== Prevalence of ambiguous 2-digit years in the full dataset ===")
for field, (lo, hi) in _AMBIG.items():
    total = counts[field]['total_parseable']
    ambig = counts[field]['ambiguous']
    pct   = 100 * ambig / total if total else 0.0
    print(f"  {field}")
    print(f"    Total records with a date value : {total:>8,}")
    print(f"    Ambiguous yy {lo:02d}–{hi:02d}           : {ambig:>8,}  ({pct:.1f}%)")
    print()


=== ApplicantBirthDate — ambiguous range yy 34–54 ===
    yy    1800s    1900s    total   % 1800s  level
    34        0     9821     9821      0.0%  ~~~
    35        0    10219    10219      0.0%  ~~~
    36        0     9402     9402      0.0%  ~~~
    37        0     9629     9629      0.0%  ~~~
    38        0     8874     8874      0.0%  ~~~
    39        0     7224     7224      0.0%  ~~~
    40        0     6218     6218      0.0%  ~~~
    41        0     5286     5286      0.0%  ~~~
    42        1     4113     4114      0.0%  ~~~
    43        0     2870     2870      0.0%  ~~~
    44        0     1825     1825      0.0%  ~~~
    45        0      918      918      0.0%  ~~~
    46        1      819      820      0.1%  ~~~
    47        0      712      712      0.0%  ~~~
    48        0      530      530      0.0%  ~~
    49        0      438      438      0.0%  ~~
    50        1      449      450      0.2%  ~
    51        0       25       25      0.0%  ~
    52        0    

In [ ]:
# resolve_dates: three-priority resolution for all date fields

_meta_df = _load_metadata()


merge_index = _build_merge_index(BZK_DATA_DIR, _meta_df)

_DATE_FIELDS  = ['ApplicantBirthDate', 'VictimBirthDate', 'VictimDeathDate']
_SKIP_RAW_SET = {'nan', 'none', 'null', '', '-', '?'}


def resolve_dates(rec: dict) -> dict:
    """
    Resolve all three date fields for one BZK record.

    Priority per field:
      1. metadata       — from merge_index (linked via name+birthdate)
      2. filename       — date encoded in filename 
      3. normalized     — normalize_date() on the raw OCR value
      4. unparsed       — raw OCR value present but unparseable
      5. missing        — raw OCR value absent (OCR failure or blank on card)
      6. not_applicable — field structurally absent for this layout class
    """
    fname        = rec.get('filename', '')
    hit          = merge_index.get(fname)
    fi           = filename_index.get(fname)
    layout_class = fi[1] if fi else None
    ctx          = {'layout_class': layout_class}
    absent       = _LAYOUT_ABSENT_FIELDS.get(layout_class, set())
    result       = {}

    for field in _DATE_FIELDS:
        value = source = None
        uncertainty = 0

        # Priority 1 — metadata (VictimDeathDate only; birth dates excluded — circular join)
        if hit and field == 'VictimDeathDate':
            sd = hit.get('meta_sd_iso')
            if sd: value, source = sd, 'metadata'

        # Priority 2 — filename
        if value is None and fi is not None:
            gt_iso, _lc, fname_field = fi
            if fname_field == field:
                value, source = gt_iso, 'filename'

        # Priority 3 — normalization pipeline
        if value is None:
            raw = (rec.get(field) or '').strip()
            if raw and raw.lower() not in _SKIP_RAW_SET:
                norm, uncertainty = normalize_date(raw, col=field, ctx=ctx)
                value, source = (norm, 'normalized') if norm else ('', 'unparsed')
            elif field in absent:
                value, source = '', 'not_applicable'
            else:
                value, source = '', 'missing'

        result[field] = {'value': value or '', 'source': source, 'uncertainty': uncertainty}

        # Feed resolved VictimDeathDate into ctx for birth year century disambiguation
        if field == 'VictimDeathDate' and value:
            ctx['death_year'] = _extract_year(value)

    return result


Merge matches: 11,761  →  11,057 unique filenames


In [26]:
# Resolve dates for all BZK records and write output
# Output: resolved_dates.jsonl  — one record per line with resolved date fields + source tags

from collections import defaultdict
import json

OUTPUT_PATH = 'resolved_dates.jsonl'
_DATE_FIELDS = ['ApplicantBirthDate', 'VictimBirthDate', 'VictimDeathDate']

source_counts = {f: defaultdict(int) for f in _DATE_FIELDS}
n_records = 0

with open(OUTPUT_PATH, 'w', encoding='utf-8') as out:
    for _jsonl in sorted(BZK_DATA_DIR.glob('*.jsonl')):
        with open(_jsonl, encoding='utf-8') as fh:
            for line in fh:
                rec = json.loads(line)
                resolved = resolve_dates(rec)

                out_rec = {'filename': rec.get('filename', '')}
                for field in _DATE_FIELDS:
                    r = resolved[field]
                    out_rec[field]             = r['value']
                    out_rec[field + '_source']      = r['source']
                    out_rec[field + '_uncertainty'] = r['uncertainty']
                    source_counts[field][r['source']] += 1

                out.write(json.dumps(out_rec, ensure_ascii=False) + '\n')
                n_records += 1

print(f'Written {n_records:,} records → {OUTPUT_PATH}')
print()

# Source breakdown per field
all_sources = ['metadata', 'filename', 'normalized', 'unparsed', 'missing', 'not_applicable']
col_w = 16
print(f"{'Field':<24} " + '  '.join(f'{s:>{col_w}}' for s in all_sources))
print('-' * (24 + (col_w + 2) * len(all_sources)))
for field in _DATE_FIELDS:
    total = sum(source_counts[field].values()) or 1
    row = f'{field:<24} '
    for s in all_sources:
        n = source_counts[field][s]
        row += f'  {f"{n:,} ({100*n/total:.0f}%)":>{col_w}}'
    print(row)


Written 1,911,042 records → resolved_dates.jsonl

Field                            metadata          filename        normalized          unparsed           missing    not_applicable
------------------------------------------------------------------------------------------------------------------------------------
ApplicantBirthDate                   0 (0%)   1,096,750 (57%)     629,727 (33%)          152 (0%)     184,372 (10%)           41 (0%)
VictimBirthDate                      0 (0%)       14,241 (1%)     470,101 (25%)          695 (0%)   1,389,257 (73%)       36,748 (2%)
VictimDeathDate                  7,472 (0%)            0 (0%)     350,404 (18%)        1,937 (0%)   1,551,229 (81%)            0 (0%)


In [ ]:
# Quality note on source reliability
# metadata  - external archival data source : high trust
# filename  — date encoded in the scan filename : high trust 
# normalized — date parsed from OCR-extracted text via normalize_date (medium trust, may carry uncertainty flags)
# unparsed  — OCR value present but could not be parsed              (no date recovered)
# missing   — no OCR value present in the record                     (no date recovered)

# Check a sample of resolved records
import json
sample = []
with open('resolved_dates.jsonl') as f:
    for i, line in enumerate(f):
        #if i >= 10: break
        sample.append(json.loads(line))

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.DataFrame(sample)


,filename,ApplicantBirthDate,ApplicantBirthDate_source,ApplicantBirthDate_uncertainty,VictimBirthDate,VictimBirthDate_source,VictimBirthDate_uncertainty,VictimDeathDate,VictimDeathDate_source,VictimDeathDate_uncertainty
0,1818_06_07_1_0.jpg,1818-06-07,normalized,0,1868-06-06,normalized,0,,missing,0
1,1819_02_20_1_0.jpg,1891-02-20,normalized,0,,missing,0,,missing,0
2,1819_04_17_1_0.jpg,1819-04-17,normalized,0,,missing,0,,missing,0
3,1819_06_06_1_0.jpg,1819-06-06,filename,0,,missing,0,,missing,0
4,1819_04_19_geprüft_1_0.jpg,1899-04-19,normalized,0,,missing,0,,missing,0
...,...,...,...,...,...,...,...,...,...,...
1911037,Vereine u. Musikveeinigungen_466_0.jpg,,missing,0,1946-01-01,normalized,0,,missing,0
1911038,Vereinigungen_439_0.jpg,,missing,0,,missing,0,,missing,0
1911039,Ärzte_2_0.jpg,,missing,0,,missing,0,,missing,0
1911040,Ärzte_3_0.jpg,1869-11-22,normalized,0,,missing,0,,missing,0
